# Ingestão de Dados Hidrológicos Projeto Iguaçu
## Hidrovia Paraguai-Paraná (Estação: Ladário)

Notebook responsável pela extração dos dados de níveis fluviais da Estação de Ladário da Hidrovia do Rio Paraguai. 

Os dados são obtidos via API da Agência Nacional de Águas e Saneamento Básico (ANA) através da [API](https://www.ana.gov.br/hidrowebservice/swagger-ui/index.html#/WSEstacoesTelemetricasController/oAUth) para a Estação de Ladário (Código Estação: 66825000).

Destino:
- Parquet - Volume:
  - Ladário: `01_bronze/ana/ladario`
- Tabela:
  - Ladário: `01_bronze.ana_ladario`

Schema:

```python
root
 |-- Cota_01: float (nullable = true)
 |-- Cota_02: float (nullable = true)
 |-- Cota_03: float (nullable = true)
 |-- Cota_04: float (nullable = true)
 |-- Cota_05: float (nullable = true)
 |-- Cota_06: float (nullable = true)
 |-- Cota_07: float (nullable = true)
 |-- Cota_08: float (nullable = true)
 |-- Cota_09: float (nullable = true)
 |-- Cota_10: float (nullable = true)
 |-- Cota_11: float (nullable = true)
 |-- Cota_12: float (nullable = true)
 |-- Cota_13: float (nullable = true)
 |-- Cota_14: float (nullable = true)
 |-- Cota_15: float (nullable = true)
 |-- Cota_16: float (nullable = true)
 |-- Cota_17: float (nullable = true)
 |-- Cota_18: float (nullable = true)
 |-- Cota_19: float (nullable = true)
 |-- Cota_20: float (nullable = true)
 |-- Cota_21: float (nullable = true)
 |-- Cota_22: float (nullable = true)
 |-- Cota_23: float (nullable = true)
 |-- Cota_24: float (nullable = true)
 |-- Cota_25: float (nullable = true)
 |-- Cota_26: float (nullable = true)
 |-- Cota_27: float (nullable = true)
 |-- Cota_28: float (nullable = true)
 |-- Cota_29: float (nullable = true)
 |-- Cota_30: float (nullable = true)
 |-- Cota_31: float (nullable = true)
 |-- Data_Hora_Dado: timestamp (nullable = true)
 |-- Dia_Maxima: integer (nullable = true)
 |-- Dia_Minima: integer (nullable = true)
 |-- Maxima: float (nullable = true)
 |-- Minima: float (nullable = true)
 |-- Media: float (nullable = true)
 |-- month: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- ingestion_date: date (nullable = false)
```

In [0]:
## libs python
import os
import requests
import pandas as pd

from dotenv import load_dotenv

In [0]:
## libs pyspark
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType, IntegerType

In [0]:
load_dotenv('env')

In [0]:
# Constrantes
IDENTIFY = os.getenv('IDENTIFY') #os.environ['IDENTIFY']
PASSWORD = os.getenv('PASSWORD') #os.environ['PASSWORD']

STATION_CODE = '66825000'

ANA_API_BASE_URL = 'https://www.ana.gov.br/hidrowebservice'
OAUTH_URL = f'{ANA_API_BASE_URL}/EstacoesTelemetricas/OAUth/v1'
COTAS_URL = f'{ANA_API_BASE_URL}/EstacoesTelemetricas/HidroSerieCotas/v1'

ladario_parquet_path = '/Volumes/iguacu_lakehouse/01_bronze/ana/ladario'
ladario_table_name = 'iguacu_lakehouse.01_bronze.ana_ladario'

In [0]:
## informacoes (colunas) da API
cotas_columns = [
    'Cota_01',
    'Cota_02',
    'Cota_03',
    'Cota_04',
    'Cota_05',
    'Cota_06',
    'Cota_07',
    'Cota_08',
    'Cota_09',
    'Cota_10',
    'Cota_11',
    'Cota_12',
    'Cota_13',
    'Cota_14',
    'Cota_15',
    'Cota_16',
    'Cota_17',
    'Cota_18',
    'Cota_19',
    'Cota_20',
    'Cota_21',
    'Cota_22',
    'Cota_23',
    'Cota_24',
    'Cota_25',
    'Cota_26',
    'Cota_27',
    'Cota_28',
    'Cota_29',
    'Cota_30',
    'Cota_31'
]

others_columns = [
    'Data_Hora_Dado',
    'Dia_Maxima', 
    'Dia_Minima',
    'Maxima',
    'Minima',
    'Media'
]

In [0]:
def get_token(identifier: str, password: str):

    headers = {
        "accept": "*/*",
        "Identificador": identifier,
        "Senha": password,
    }

    response = requests.get(OAUTH_URL, headers=headers)
    if response.status_code == 200:
        data = response.json()
        token = data['items']['tokenautenticacao']

        return token
    else:
        print('Erro na requisição:', response.status_code)

    return None

In [0]:
def get_cotas(token: str, station_code: str, year: str):
    
    params = {
        "Código da Estação": station_code,
        "Tipo Filtro Data": "DATA_LEITURA",
        "Data Inicial (yyyy-MM-dd)": f'{year}-01-01',
        "Data Final (yyyy-MM-dd)": f'{year}-12-31'
    }

    headers = {
        "accept": "*/*",
        "Authorization": f"Bearer {token}",
    }

    response = requests.get(COTAS_URL, params=params, headers=headers)
    if response.status_code == 200:
        data = response.json()
        cotas = data['items']

        cotas_df = pd.DataFrame(cotas)
        cotas_df = cotas_df[cotas_columns + others_columns]

        spark_df = spark.createDataFrame(cotas_df).withColumn(
            'Data_Hora_Dado',
            F.to_timestamp(F.col('Data_Hora_Dado'), 'yyyy-MM-dd HH:mm:ss.S')
        ).withColumn(
            'month',
            F.month(F.col('Data_Hora_Dado'))
        ).withColumn(
            'year',
            F.year(F.col('Data_Hora_Dado'))
        ).withColumn(
            'ingestion_date',
            F.current_date()
        )

        float_columns = [
            'Cota_01','Cota_02','Cota_03','Cota_04','Cota_05','Cota_06','Cota_07','Cota_08','Cota_09','Cota_10','Cota_11','Cota_12','Cota_13','Cota_14','Cota_15','Cota_16','Cota_17','Cota_18','Cota_19','Cota_20','Cota_21','Cota_22','Cota_23','Cota_24','Cota_25','Cota_26','Cota_27','Cota_28','Cota_29','Cota_30','Cota_31','Maxima','Minima','Media'
        ]

        for col in float_columns:
            spark_df = spark_df.withColumn(col, F.col(col).cast(FloatType()))
        
        for col in ['Dia_Maxima','Dia_Minima']:
            spark_df = spark_df.withColumn(col, F.col(col).cast(IntegerType()))

        return spark_df

    else:
        print('Erro na requisição:', response.status_code)

    return None

In [0]:
token = get_token(IDENTIFY, PASSWORD)

for year in ['2020', '2021', '2022', '2023', '2024', '2025']:
    ladario_df = get_cotas(token, STATION_CODE, year)

    #salvar arquivo parquet
    (
        ladario_df
        .write
        .partitionBy(['year', 'month'])
        .mode('append')
        .format('parquet')
        .save(ladario_parquet_path)
    )
    
    #salvar tabela delta
    (
        ladario_df
        .write
        .format('delta')
        .mode('append')
        .partitionBy(['year', 'month'])
        .saveAsTable(ladario_table_name)
    )

In [0]:
%sql

SELECT 
  * 
FROM iguacu_lakehouse.01_bronze.ana_ladario 
LIMIT 5